In [1]:
# -*- coding: utf-8 -*-
"""main_v3_dual_evaluator.py

LangGraph Joke Generation Pipeline with Dual Evaluation (GPT-OSS + Mistral)
"""

# !pip uninstall -y transformers accelerate peft bitsandbytes trl
# !pip install -q -U bitsandbytes accelerate peft transformers trl datasets
# !pip install faiss-cpu

import os
import json
import torch
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

import wandb
from huggingface_hub import notebook_login

In [ ]:
"""
Setup environment and imports
"""

print("Setting up environment...")

# Check for GPU
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")
else:
    print("No GPU detected. Will use CPU (slower)")

print(" Environment setup complete\n")

"""
LangGraph Joke Generation Pipeline - Orchestrator with Dual Evaluation
Imports and orchestrates existing implementations
"""

"""
LangGraph Joke Generation Pipeline - Orchestrator
Imports and orchestrates existing implementations
"""
from typing import TypedDict, Literal, List, Dict
from langgraph.graph import StateGraph, END
from pathlib import Path

import sys

#  set project root = folder c:\Users\Anwender\humor-project
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# if you run from a notebook inside some subfolder, walk up until we find "src"
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))         # so "import src...." works
sys.path.insert(0, str(PROJECT_ROOT / "src")) # so "import evaluation...." works if you want
print("Using PROJECT_ROOT:", PROJECT_ROOT)
# Import your existing implementations
from using_llama_with_rag import RAGHumorGenerator, RAGConfig
from gpt_joke_generation import GPT4JokeGenerator
from src.evaluation.llm_evaluator import LLMHumorScorer
from src.evaluation.mistral_evaluator import MistralHumorScorer

print(" All modules imported successfully\n")

In [ ]:

# ============================================================================
# LANGGRAPH STATE DEFINITION
# ============================================================================

class JokeGenerationState(TypedDict):
    """State for the joke generation pipeline"""
    # Input
    word1: str
    word2: str

    # Generation tracking
    llama_jokes: List[str]
    gpt_jokes: List[str]
    candidate_jokes: List[str]
    joke_sources: List[str]      # "Llama" or "GPT" per joke

    # Scoring tracking (scores = Mistral normalized, used for ranking/threshold)
    scores: List[float]          # Mistral normalized 0-10 (primary ranking)
    final_scores: List[float]    # Ensemble GPT 60% + Mistral 40% (reference only)
    gpt_scores: List[float]      # GPT adjusted scores
    mistral_scores: List[float]  # Mistral raw 0-4

    # Best result tracking (selected by Mistral score)
    best_joke: str
    best_score: float

    # Iteration tracking
    iteration_count: int
    max_iterations: int

    # History for debugging
    history: List[Dict]


In [ ]:

# ============================================================================
# NODE FUNCTIONS
# ============================================================================

def generate_jokes_llama(state: JokeGenerationState, llama_gen: RAGHumorGenerator) -> Dict:
    """Generate jokes using Llama model"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    mode = "headline" if not word2 else "word-pair"
    print(f"\n[Iteration {iteration + 1}] Generating jokes with Llama ({mode} mode)...")

    result_with_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=True,  # with RAG
        num_candidates=3
    )

    result_without_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=False,  # No RAG - just fine-tuned model
        num_candidates=3
    )

    jokes = result_with_RAG["generated_jokes"] + result_without_RAG["generated_jokes"]

    print('result_with_RAG jokes:')
    print(result_with_RAG["generated_jokes"])

    print('result_without_RAG jokes:')
    print(result_without_RAG["generated_jokes"])

    print(f"   Generated {len(jokes)} Llama jokes")

    return {"llama_jokes": jokes}


def generate_jokes_gpt(state: JokeGenerationState, gpt_gen: GPT4JokeGenerator) -> Dict:
    """Generate jokes using GPT-OSS (reduced to 2 to limit self-bias)"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    mode = "headline" if not word2 else "word-pair"
    print(f"\n[Iteration {iteration + 1}] Generating jokes with GPT-OSS ({mode} mode)...")

    jokes = gpt_gen.generate_jokes(word1, word2, num_jokes=2, verbose=False)

    print(f"   Generated {len(jokes)} GPT-OSS jokes")

    return {"gpt_jokes": jokes}


def combine_jokes(state: JokeGenerationState) -> JokeGenerationState:
    """Combine jokes from both generators and track sources"""
    llama_jokes = state.get("llama_jokes", [])
    gpt_jokes = state.get("gpt_jokes", [])

    all_jokes = llama_jokes + gpt_jokes
    sources = ["Llama"] * len(llama_jokes) + ["GPT"] * len(gpt_jokes)

    print(f"\nCombined {len(all_jokes)} total jokes ({len(llama_jokes)} Llama + {len(gpt_jokes)} GPT)")

    return {
        **state,
        "candidate_jokes": all_jokes,
        "joke_sources": sources,
    }


def normalize_mistral_score(raw: float) -> float:
    """
    Normalize Mistral humor score (0-4) to 0-10 scale using
    percentile-aware mapping based on training data distribution.
    
    Training data stats (hahackathon_train.csv):
      - Range: 0.1 - 4.0, Mean: 2.26, SD: 0.57
      - ~90% of scores fall between 1.5 and 3.0
      - Score 3.0+ = top quartile (genuinely funny)
      - Score 1.5-  = bottom quartile (not funny)
    
    We use a piecewise linear mapping that stretches the meaningful
    range (1.0-3.5) across most of the 0-10 scale:
      Mistral 0.0 -> 0.0    (no humor)
      Mistral 1.0 -> 2.0    (weak humor)
      Mistral 2.0 -> 4.5    (below average)
      Mistral 2.26-> 5.0    (average - matches training mean)
      Mistral 3.0 -> 7.0    (good humor)
      Mistral 3.5 -> 8.5    (very good)
      Mistral 4.0 -> 10.0   (exceptional)
    """
    raw = max(0.0, min(4.0, raw))
    
    # Piecewise linear mapping
    breakpoints = [(0.0, 0.0), (1.0, 2.0), (2.0, 4.5), (2.26, 5.0),
                   (3.0, 7.0), (3.5, 8.5), (4.0, 10.0)]
    
    for i in range(len(breakpoints) - 1):
        x0, y0 = breakpoints[i]
        x1, y1 = breakpoints[i + 1]
        if raw <= x1:
            t = (raw - x0) / (x1 - x0)
            return y0 + t * (y1 - y0)
    
    return 10.0


def score_jokes_dual(
    state: JokeGenerationState,
    llm_scorer: LLMHumorScorer,
    mistral_scorer: MistralHumorScorer
) -> JokeGenerationState:
    """
    Score jokes using dual evaluators.
    
    RANKING BY MISTRAL ONLY (trained on human humor ratings):
    - Mistral (0-4, normalized to 0-10) drives best-joke selection & threshold.
    - LLM scorer (1-10, multi-criteria) kept as reference in final_scores ensemble.
    
    LLM self-bias discount: LLM scores multiplied by 0.80 for GPT-generated jokes.
    """
    jokes = state["candidate_jokes"]
    word1 = state["word1"]
    word2 = state["word2"]
    sources = state.get("joke_sources", [])
    iteration = state.get("iteration_count", 0)

    print(f"\n[Iteration {iteration + 1}] Scoring {len(jokes)} jokes with LLM + Mistral...")

    # --- LLM scorer: multi-criteria quality (1-10) ---
    print("   Scoring with LLM scorer (batch mode - single API call)...")
    llm_scores_raw = llm_scorer.score_jokes(jokes, word1, word2, verbose=False)

    # Apply self-bias discount for GPT-generated jokes
    LLM_SELF_BIAS_DISCOUNT = 0.80
    llm_scores = []
    for i, score in enumerate(llm_scores_raw):
        src = sources[i] if i < len(sources) else "?"
        if src == "GPT":
            llm_scores.append(score * LLM_SELF_BIAS_DISCOUNT)
        else:
            llm_scores.append(score)

    # --- Mistral: pure humor rating (0-4) ---
    print("   Scoring with Mistral (humor funniness only)...")
    mistral_scores = mistral_scorer.score_jokes(jokes, word1, word2, verbose=False)
    
    # Normalize Mistral to 0-10 using distribution-aware mapping
    mistral_scores_normalized = [normalize_mistral_score(s) for s in mistral_scores]
    
    # Ensemble (reference only, NOT used for ranking)
    W_LLM = 0.6
    W_MISTRAL = 0.4
    ensemble_scores = [
        W_LLM * llm + W_MISTRAL * mistral
        for llm, mistral in zip(llm_scores, mistral_scores_normalized)
    ]

    # Print score breakdown - sorted by Mistral (primary ranking)
    print(f"\n   Score breakdown (RANKED BY MISTRAL, ensemble LLM {W_LLM:.0%} + Mistral {W_MISTRAL:.0%} = reference):")
    print(f"   {'='*130}")
    print(f"   {'#':>3} {'Src':<6} {'Mistral':>7} {'(norm)':>7}  {'LLM_raw':>7} {'LLM_adj':>7}  {'Ensembl':>7}  Joke")
    print(f"   {'-'*130}")
    
    # Display sorted by Mistral normalized (descending)
    display_order = sorted(range(len(jokes)), key=lambda i: mistral_scores_normalized[i], reverse=True)
    for rank, i in enumerate(display_order, 1):
        joke = jokes[i]
        llm_raw = llm_scores_raw[i]
        llm_adj = llm_scores[i]
        m_raw = mistral_scores[i]
        m_norm = mistral_scores_normalized[i]
        ens = ensemble_scores[i]
        src = sources[i] if i < len(sources) else "?"
        disc = "*" if src == "GPT" else " "
        marker = " << BEST" if rank == 1 else ""
        print(f"   {rank:3d} {src:<6} {m_raw:5.2f}/4 ({m_norm:5.2f})  {llm_raw:5.2f}/10 {llm_adj:5.2f}{disc}   {ens:5.2f}/10  {joke[:55]}...{marker}")

    # Find best joke BY MISTRAL normalized score
    if mistral_scores_normalized:
        best_idx = mistral_scores_normalized.index(max(mistral_scores_normalized))
        best_joke = jokes[best_idx]
        best_score = mistral_scores_normalized[best_idx]
    else:
        best_joke = ""
        best_score = 0.0

    print(f"\n   * = LLM score discounted by {LLM_SELF_BIAS_DISCOUNT} (GPT-generated joke)")
    print(f"\n   Best (by Mistral): {best_score:.2f}/10  (raw: {mistral_scores[best_idx]:.2f}/4, LLM: {llm_scores[best_idx]:.2f}, Ensemble: {ensemble_scores[best_idx]:.2f})")
    print(f"   Avg Mistral norm:  {sum(mistral_scores_normalized)/len(mistral_scores_normalized):.2f}/10")
    print(f"   Range:             {min(mistral_scores_normalized):.2f} - {max(mistral_scores_normalized):.2f}")

    return {
        **state,
        "scores": mistral_scores_normalized,    # PRIMARY: drives ranking & threshold
        "final_scores": ensemble_scores,         # REFERENCE: ensemble for comparison
        "gpt_scores": llm_scores,
        "mistral_scores": mistral_scores,
        "best_joke": best_joke,
        "best_score": best_score
    }


def check_score_threshold(state: JokeGenerationState) -> Literal["refine", "output"]:
    """Decision node: check if Mistral score meets threshold"""
    score = state["best_score"]
    iteration = state.get("iteration_count", 0)
    max_iter = state.get("max_iterations", 3)

    print(f"\nDecision: Mistral={score:.2f}/10, Iteration={iteration + 1}/{max_iter}")

    if score >= 7.0:
        print(f"   Mistral threshold met! (>=7.0 = raw 3.0/4)")
        return "output"
    elif iteration >= max_iter - 1:
        print(f"   Max iterations reached, outputting best result")
        return "output"
    else:
        print(f"   Mistral below threshold, refining...")
        return "refine"


def increment_iteration(state: JokeGenerationState) -> JokeGenerationState:
    """Increment iteration counter and save history"""
    current_iter = state.get("iteration_count", 0)
    history = state.get("history", [])

    # Save this iteration's results
    history.append({
        "iteration": current_iter + 1,
        "best_score": state["best_score"],
        "best_joke": state["best_joke"],
        "num_candidates": len(state.get("candidate_jokes", [])),
        "avg_mistral_norm": sum(state.get("scores", [0])) / max(len(state.get("scores", [1])), 1),
        "avg_llm_score": sum(state.get("gpt_scores", [0])) / max(len(state.get("gpt_scores", [1])), 1),
        "avg_mistral_raw": sum(state.get("mistral_scores", [0])) / max(len(state.get("mistral_scores", [1])), 1)
    })

    return {
        **state,
        "iteration_count": current_iter + 1,
        "history": history
    }


def output_result(state: JokeGenerationState) -> JokeGenerationState:
    """Final output node - shows best and second-best joke ranked by Mistral"""
    print(f"\n{'='*80}")
    print(f"FINAL RESULT (ranked by Mistral)")
    print(f"{'='*80}")

    word1 = state['word1']
    word2 = state['word2']
    if word2:
        print(f"\nWord Pair: '{word1}' + '{word2}'")
    else:
        print(f"\nHeadline: '{word1}'")
    print(f"Total Iterations: {state.get('iteration_count', 0) + 1}")

    # Rank all jokes by Mistral normalized score (= scores)
    jokes = state.get("candidate_jokes", [])
    m_norm = state.get("scores", [])          # Mistral normalized
    ens_sc = state.get("final_scores", [])    # Ensemble reference
    llm_sc = state.get("gpt_scores", [])
    mis_sc = state.get("mistral_scores", [])  # Mistral raw
    sources = state.get("joke_sources", [])

    if jokes and m_norm:
        ranked = sorted(
            zip(jokes, m_norm, ens_sc, llm_sc, mis_sc, sources + ["?"] * max(0, len(jokes) - len(sources))),
            key=lambda x: x[1], reverse=True
        )

        # Best joke
        j1, mn1, e1, g1, m1, s1 = ranked[0]
        print(f"\n  #1 BEST  [Mistral:{mn1:.2f}/10 raw:{m1:.2f}/4]  (LLM:{g1:.1f}, Ens:{e1:.2f})  Source:{s1}")
        print(f"  {j1}")

        # Second best
        if len(ranked) >= 2:
            j2, mn2, e2, g2, m2, s2 = ranked[1]
            print(f"\n  #2       [Mistral:{mn2:.2f}/10 raw:{m2:.2f}/4]  (LLM:{g2:.1f}, Ens:{e2:.2f})  Source:{s2}")
            print(f"  {j2}")
    else:
        print(f"\n  (no jokes scored)")

    print(f"\n{'='*80}\n")

    return state

In [ ]:

# ============================================================================
# GRAPH BUILDER
# ============================================================================

def build_joke_generation_graph(
    llama_generator: RAGHumorGenerator,
    gpt_generator: GPT4JokeGenerator,
    llm_scorer: LLMHumorScorer,
    mistral_scorer: MistralHumorScorer
):
    """
    Build the LangGraph for joke generation pipeline with dual evaluation.

    Args:
        llama_generator: RAGHumorGenerator
        gpt_generator: GPT4JokeGenerator
        llm_scorer: LLMHumorScorer (experimental, reference only)
        mistral_scorer: MistralHumorScorer (primary ranking)

    Returns:
        Compiled LangGraph
    """
    workflow = StateGraph(JokeGenerationState)

    # Add nodes with bound generators/scorers
    workflow.add_node(
        "start",
        lambda state: state  # pass-through to fan out to both generators
    )
    workflow.add_node(
        "generate_llama",
        lambda state: generate_jokes_llama(state, llama_generator)
    )
    workflow.add_node(
        "generate_gpt",
        lambda state: generate_jokes_gpt(state, gpt_generator)
    )
    workflow.add_node("combine_jokes", combine_jokes)
    workflow.add_node(
        "score_jokes",
        lambda state: score_jokes_dual(state, llm_scorer, mistral_scorer)
    )
    workflow.add_node("increment_iteration", increment_iteration)
    workflow.add_node("output", output_result)

    # Fan-out: start node dispatches to both generators in parallel
    workflow.set_entry_point("start")
    workflow.add_edge("start", "generate_llama")
    workflow.add_edge("start", "generate_gpt")

    # Add edges - both generators feed into combine
    workflow.add_edge("generate_llama", "combine_jokes")
    workflow.add_edge("generate_gpt", "combine_jokes")

    # After combining, score the jokes
    workflow.add_edge("combine_jokes", "score_jokes")

    # Decision point: refine or output
    workflow.add_conditional_edges(
        "score_jokes",
        check_score_threshold,
        {
            "refine": "increment_iteration",
            "output": "output"
        }
    )

    # Refinement loop - go back to start (fans out to both generators)
    workflow.add_edge("increment_iteration", "start")

    # End the workflow
    workflow.add_edge("output", END)

    return workflow.compile()

In [ ]:
# ============================================================================
# LOAD MODELS (run once, then skip when re-running pipeline)
# ============================================================================

print("="*80)
print("LANGGRAPH JOKE GENERATION PIPELINE")
print("Mistral primary ranking | LLM scorer as reference")
print("="*80)
print()

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found in environment. Please set it in .env file")

config = RAGConfig()
config.TEMPERATURE = 0.65
config.TOP_K = 50

print("Initializing components...\n")

# 1. Llama Generator
try:
    llama_gen = RAGHumorGenerator(config)
    llama_gen.load_model()
    print("Llama generator initialized\n")
except Exception as e:
    print(f"Could not load Llama model: {e}\n")
    llama_gen = None

# 2. GPT-OSS Generator
gpt_gen = GPT4JokeGenerator(
    api_key=NVIDIA_API_KEY,
    temperature=0.8,
    max_retries=1
)
print("GPT-OSS generator initialized\n")

# 3. LLM Scorer (experimental, reference only)
llm_scorer = LLMHumorScorer(
    api_key=NVIDIA_API_KEY,
    temperature=0.4,
    max_retries=2
)
print("LLM scorer initialized (experimental, reference only)\n")

# 4. Mistral Scorer (primary ranking - fine-tuned on human humor ratings)
mistral_scorer = MistralHumorScorer(
    base_model_id="mistralai/Ministral-3-8B-Base-2512",
    num_labels=2,
    max_length=512
)
print(f"Mistral checkpoint: {mistral_scorer.checkpoint_path}")

if not mistral_scorer.checkpoint_path.exists():
    print(f"Mistral checkpoint not found! Expected: {mistral_scorer.checkpoint_path}")
    mistral_scorer = None
else:
    mistral_scorer.load_model()
    print("Mistral scorer initialized (PRIMARY)\n")

print("="*80)
print("All models loaded.")
print("="*80)

In [ ]:
# ============================================================================
# BUILD RAG DATABASE & COMPILE GRAPH (run once after loading models)
# ============================================================================

# Load RAG database
if llama_gen is not None:
    llama_gen.load_rag_database(build_if_missing=True)
    print("RAG database loaded\n")

# Build the graph
if mistral_scorer is None:
    raise RuntimeError("Cannot run pipeline without Mistral scorer (primary)")
if llama_gen is None:
    raise RuntimeError("Cannot run pipeline without Llama generator")

graph = build_joke_generation_graph(llama_gen, gpt_gen, llm_scorer, mistral_scorer)
print("LangGraph compiled (Mistral primary, LLM reference)")

In [ ]:

# ============================================================================
# RUN PIPELINE (re-run this cell to regenerate jokes)
# ============================================================================
import csv
from datetime import datetime

# Supports two input modes:
#   ("word1", "word2")  -> word-pair mode: joke connecting both words
#   ("headline text", "") -> headline mode: funny commentary on headline
test_pairs = [
    ("banana", "satellite"),
    ("angry", "teacup"),
    ("cosmic", "toaster"),
    ("sleepy", "volcano"),
    ("neon", "hamster"),
    ("awkward", "penguin"),
    ("electric", "spaghetti"),
    ("dramatic", "cactus"),
    ("confused", "astronaut"),
    ("rubber", "tornado"),
    ("invisible", "taco"),
    ("sarcastic", "dolphin"),
    ("haunted", "calculator"),
    ("spicy", "cloud"),
    ("polite", "chainsaw"),
    ("disco", "librarian"),
    ("lonely", "robot"),
    ("caffeinated", "squirrel"),
    ("confused", "mailbox"),
    # Headline mode examples (word2 = ""):
    # ("Scientists discover that cats have been faking sleep for centuries", ""),
    # ("Local man wins lottery, immediately buys 10000 rubber ducks", ""),
]

results_summary = []

# --- Open CSVs BEFORE the loop so results are saved incrementally ---
output_dir = PROJECT_ROOT / "output"
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_csv_path = output_dir / f"best_jokes_{timestamp}.csv"
all_csv_path  = output_dir / f"all_jokes_{timestamp}.csv"

best_csv_file = open(best_csv_path, 'w', newline='', encoding='utf-8')
all_csv_file  = open(all_csv_path,  'w', newline='', encoding='utf-8')

best_writer = csv.writer(best_csv_file)
all_writer  = csv.writer(all_csv_file)

best_writer.writerow(["#", "Word1", "Word2", "Best_Joke", "Best_Source", "2nd_Best_Joke", "2nd_Source",
                       "Best_Mistral_0to4", "Best_Mistral_Norm", "Best_GPT", "Best_Ensemble",
                       "2nd_Mistral_0to4",  "2nd_Mistral_Norm",  "2nd_GPT",  "2nd_Ensemble", "Human_Rating"])
all_writer.writerow(["#", "PairIdx", "Rank", "Word1", "Word2", "Joke", "Source",
                      "Mistral_0to4", "Mistral_0to10", "GPT_Score", "Ensemble", "Human_Rating"])

all_row_num = 1
print(f"Incremental CSVs: {best_csv_path.name} / {all_csv_path.name}")
print(f"Ranking: Mistral-only (ensemble kept as reference)")

for pair_idx, (word1, word2) in enumerate(test_pairs, 1):
    is_headline = not word2
    if is_headline:
        print("\n" + "="*80)
        print(f"PROCESSING [{pair_idx}/{len(test_pairs)}] HEADLINE: '{word1}'")
        print("="*80)
    else:
        print("\n" + "="*80)
        print(f"PROCESSING [{pair_idx}/{len(test_pairs)}]: '{word1}' + '{word2}'")
        print("="*80)

    initial_state = {
        "word1": word1, "word2": word2,
        "llama_jokes": [], "gpt_jokes": [], "candidate_jokes": [],
        "joke_sources": [],
        "scores": [], "final_scores": [], "gpt_scores": [], "mistral_scores": [],
        "best_joke": "", "best_score": 0.0,
        "iteration_count": 0, "max_iterations": 1, "history": []
    }

    try:
        result = graph.invoke(initial_state)
        total_candidates = len(result.get('candidate_jokes', []))

        label = f"Headline: {word1}" if is_headline else f"{word1} + {word2}"
        r = {
            "word_pair": label,
            "word1": word1,
            "word2": word2,
            "best_score": result['best_score'],
            "best_joke": result['best_joke'],
            "all_jokes": result.get('candidate_jokes', []),
            "all_scores": result.get('scores', []),            # Mistral normalized
            "final_scores": result.get('final_scores', []),    # Ensemble reference
            "gpt_scores": result.get('gpt_scores', []),
            "mistral_scores": result.get('mistral_scores', []),
            "joke_sources": result.get('joke_sources', []),
            "iterations": result.get('iteration_count', 0) + 1,
            "total_candidates": total_candidates
        }
        results_summary.append(r)

        # --- Write to CSVs immediately after each word pair ---
        sources = r['joke_sources']
        # all_scores = Mistral normalized, final_scores = ensemble
        jd = list(zip(r['all_jokes'], r['all_scores'], r['final_scores'],
                       r['gpt_scores'], r['mistral_scores'],
                       sources + ["?"] * max(0, len(r['all_jokes']) - len(sources))))
        # Sort by Mistral normalized (primary ranking)
        jd.sort(key=lambda x: x[1], reverse=True)

        # Best + 2nd best row
        if len(jd) >= 2:
            j1, mn1, e1, g1, m1, s1 = jd[0]
            j2, mn2, e2, g2, m2, s2 = jd[1]
        elif len(jd) == 1:
            j1, mn1, e1, g1, m1, s1 = jd[0]
            j2, mn2, e2, g2, m2, s2 = "", 0, 0, 0, 0, ""
        else:
            j1, mn1, e1, g1, m1, s1 = "", 0, 0, 0, 0, ""
            j2, mn2, e2, g2, m2, s2 = "", 0, 0, 0, 0, ""

        best_writer.writerow([
            pair_idx, word1, word2,
            j1, s1, j2, s2,
            f"{m1:.2f}", f"{mn1:.2f}", f"{g1:.2f}", f"{e1:.2f}",
            f"{m2:.2f}", f"{mn2:.2f}", f"{g2:.2f}", f"{e2:.2f}", ""
        ])
        best_csv_file.flush()

        # All jokes for this pair (sorted by Mistral)
        for rank, (joke, m_norm, ens, gpt, mistral, src) in enumerate(jd, 1):
            all_writer.writerow([
                all_row_num, pair_idx, rank, word1, word2, joke, src,
                f"{mistral:.2f}", f"{m_norm:.2f}", f"{gpt:.2f}", f"{ens:.2f}", ""
            ])
            all_row_num += 1
        all_csv_file.flush()

    except Exception as e:
        print(f"\nError processing word pair: {e}")
        import traceback
        traceback.print_exc()

best_csv_file.close()
all_csv_file.close()

print(f"\nDone! Generated jokes for {len(results_summary)}/{len(test_pairs)} entries.")
print(f"Best+2nd CSV : {best_csv_path}")
print(f"All jokes CSV: {all_csv_path}  ({all_row_num - 1} jokes)")


In [ ]:

# ============================================================================
# FULL OUTPUT FOR HUMAN RATING (re-run to reformat without regenerating)
# ============================================================================

print("#"*80)
print("#" + " "*26 + "JOKES FOR HUMAN RATING" + " "*30 + "#")
print("#"*80)
print(f"\nTotal entries: {len(results_summary)}")
print(f"Total jokes generated: {sum(r['total_candidates'] for r in results_summary)}")
print(f"Ranking: MISTRAL ONLY (trained on human humor ratings)")
print(f"Reference: GPT-OSS 60% + Mistral 40% ensemble (not used for selection)")
print(f"GPT self-bias discount: 0.80 for GPT-generated jokes")
print()

for pair_idx, r in enumerate(results_summary, 1):
    print("="*80)
    if r['word2']:
        print(f"WORD PAIR {pair_idx}/{len(results_summary)}: '{r['word1']}' + '{r['word2']}'")
    else:
        print(f"HEADLINE {pair_idx}/{len(results_summary)}: '{r['word1']}'")
    print(f"Best Mistral Score: {r['best_score']:.2f}/10")
    print("="*80)

    # Sort jokes by Mistral normalized score (= all_scores)
    sources = r.get('joke_sources', [])
    joke_data = list(zip(
        r['all_jokes'], r['all_scores'], r.get('final_scores', []),
        r['gpt_scores'], r['mistral_scores'],
        sources + ["?"] * max(0, len(r['all_jokes']) - len(sources))
    ))
    joke_data.sort(key=lambda x: x[1], reverse=True)

    for j_idx, (joke, m_norm, ens, gpt, mistral, src) in enumerate(joke_data, 1):
        is_best = " << SELECTED" if joke == r['best_joke'] else ""
        print(f"\n  [{j_idx:2d}] Mistral: {m_norm:.2f}/10 (raw:{mistral:.2f}/4)  GPT:{gpt:.1f}  Ens:{ens:.2f}  Src:{src}{is_best}")
        print(f"  {'-'*70}")
        print(f"  {joke}")

    print()

# --- Summary table ---
print("="*80)
print("RESULTS SUMMARY (ranked by Mistral)")
print("="*80)
print(f"\n{'#':<4} {'Input':<35} {'Mistral':<8} {'GPT':<8} {'Ensembl':<8} {'BestSrc':<8} {'Jokes':<8}")
print("-"*85)
for i, r in enumerate(results_summary, 1):
    avg_gpt = sum(r['gpt_scores']) / len(r['gpt_scores']) if r['gpt_scores'] else 0
    avg_mis = sum(r['mistral_scores']) / len(r['mistral_scores']) if r['mistral_scores'] else 0
    avg_ens = sum(r.get('final_scores', [])) / len(r.get('final_scores', [1])) if r.get('final_scores') else 0
    label = r['word_pair'][:33]
    # Find source of best joke (by Mistral)
    sources = r.get('joke_sources', [])
    jokes = r['all_jokes']
    m_norms = r['all_scores']  # Mistral normalized
    best_src = "?"
    if jokes and m_norms:
        best_idx = m_norms.index(max(m_norms))
        best_src = sources[best_idx] if best_idx < len(sources) else "?"
    print(f"{i:<4} {label:<35} {r['best_score']:<8.2f} {avg_gpt:<8.1f} {avg_ens:<8.2f} {best_src:<8} {r['total_candidates']:<8}")

avg_score = sum(r['best_score'] for r in results_summary) / len(results_summary) if results_summary else 0
print("-"*85)
print(f"{'':4} {'AVERAGE':<35} {avg_score:<8.2f}")

# --- Save JSON (full data for programmatic access) ---
output_dir = PROJECT_ROOT / "output"
output_dir.mkdir(exist_ok=True)
json_path = output_dir / f"jokes_for_rating_{timestamp}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print(f"\nJSON saved: {json_path}")
print(f"CSVs already saved incrementally during pipeline run:")
print(f"  Best+2nd : {best_csv_path}")
print(f"  All jokes: {all_csv_path}")
print(f"\n  -> Open CSVs in Excel and fill 'Human_Rating' column (1-10)")
print(f"\n{'='*80}")
print("Pipeline complete!")
print("="*80)
